# Qwen3.8-27B W4A16 + DFlash2 on CMP 170HX

## TL;DR

**Measured:** 136.38 output tok/s mean for 256-token single-stream decode and 1,946 input tok/s at a 6,603-token prompt, repeated on three CMP 170HX cards at 180 W. This notebook preserves the clean result snapshot and is the runnable path from a fresh Ubuntu host to an editable API request.

![Measured performance](assets/performance.png)

Evidence class: **MEASURED**. Exact public receipts: [Qwen3.8-27B-CMP-170HX](https://github.com/PixelML/Qwen3.8-27B-CMP-170HX/tree/41d2c414fe0f293d77087ef18cda5896664754d6).

## Requirements

- Ubuntu 22.04, one visible CMP 170HX with 64 GiB, a working NVIDIA driver, and directed airflow.
- Python 3.12, `git`, `curl`, `jq`, build tools, and enough model-storage space for roughly 25 GiB plus caches.
- Network access to GitHub, PyPI, and Hugging Face. The pinned public artifacts do not require a committed token.
- Safety stop: do not continue at 80 °C core, 85 °C memory, an NVIDIA Xid, missing storage, or a conflicting GPU workload.

## Configure

Edit only this cell. Keep `RUN_LIVE=False` when reading the committed result; change it to `True` on the target CMP host before running all cells.

In [1]:
from pathlib import Path
import contextlib, csv, json, os, runpy, subprocess, sys, time

RUN_LIVE = os.environ.get("PIXELML_RUN_LIVE", "0") == "1"
GPU_INDEX = int(os.environ.get("PIXELML_GPU_INDEX", "0"))
PORT = int(os.environ.get("PIXELML_PORT", "18020"))
MODEL_ROOT = Path(os.environ.get("PIXELML_MODEL_ROOT", "/models/pixelml/qwen38")).resolve()
RUNTIME_ROOT = Path("/app/qwen-serving")
API_SECRET_FILE = Path(os.environ.get("PIXELML_API_SECRET_FILE", "/tmp/pixelml-qwen38-api-key"))
PROMPT = "Explain why speculative decoding can emit fewer SSE events than completion tokens."

RECIPE_DIR = Path.cwd().resolve()
if not (RECIPE_DIR / "recipe.json").exists():
    RECIPE_DIR = (RECIPE_DIR / "recipes" / "qwen3.8-27b-dflash2").resolve()
REPO_ROOT = RECIPE_DIR.parents[1]
PINS = json.loads((RECIPE_DIR / "recipe.json").read_text())
print(json.dumps({"run_live": RUN_LIVE, "gpu_index": GPU_INDEX, "port": PORT, "pins": {"model_revision": PINS["model_revision"], "runtime_pin": PINS["runtime_pin"]}}, indent=2))

{
  "run_live": false,
  "gpu_index": 0,
  "port": 18020,
  "pins": {
    "model_revision": "base 1f05c441...; fast overlay 124c14e7...; draft 4d30ec736...",
    "runtime_pin": "syv-ai/qwen38-27b-rtx3090@69ba4d0688c6ae76cb9d3c4a5c3b36445e1b040c; vLLM 0.27.1"
  }
}

## Preflight

This gate prints only public-safe fields. It refuses live work when the expected GPU memory is absent, the card is already busy, or temperatures cross the publication thresholds.

In [2]:
def run(command, **kwargs):
    return subprocess.run(command, check=True, text=True, **kwargs)

if not RUN_LIVE:
    print("RECORDED MODE — clean measured outputs below; set PIXELML_RUN_LIVE=1 on the CMP host to reproduce.")
else:
    query = run([
        "nvidia-smi", "-i", str(GPU_INDEX),
        "--query-gpu=name,memory.total,memory.used,utilization.gpu,temperature.gpu,temperature.memory,driver_version",
        "--format=csv,noheader,nounits",
    ], capture_output=True).stdout.strip()
    print(query)
    name, total, used, util, core, memory, driver = [part.strip() for part in query.split(",")]
    assert float(total) >= 64000, "expected a 64 GiB CMP 170HX"
    assert float(used) < 1024 and float(util) < 5, "GPU is not free; do not interrupt its owner"
    assert float(core) < 80 and (memory in {"N/A", "[Not Supported]"} or float(memory) < 85)
    MODEL_ROOT.mkdir(parents=True, exist_ok=True)
    free_gib = __import__("shutil").disk_usage(MODEL_ROOT).free / 2**30
    assert free_gib >= 40, f"need at least 40 GiB free, found {free_gib:.1f}"
    print("PREFLIGHT PASS")

RECORDED MODE — clean measured outputs below; set PIXELML_RUN_LIVE=1 on the CMP host to reproduce.

## Install the pinned runtime

This idempotent cell clones the exact public recipe commit and builds its Python environment. It does not start a GPU workload.

In [ ]:
if RUN_LIVE:
    install = r'''set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y python3.12 python3.12-venv python3.12-dev build-essential patch git curl jq ca-certificates
sudo install -d -m 0755 /app
if [ ! -d /app/qwen-serving/.git ]; then
  sudo git clone https://github.com/syv-ai/qwen38-27b-rtx3090 /app/qwen-serving
  sudo chown -R "$USER":"$USER" /app/qwen-serving
fi
git -C /app/qwen-serving fetch origin 69ba4d0688c6ae76cb9d3c4a5c3b36445e1b040c
git -C /app/qwen-serving checkout --detach 69ba4d0688c6ae76cb9d3c4a5c3b36445e1b040c
cd /app/qwen-serving
if [ ! -x venv/bin/python ]; then
  python3.12 -m venv venv
  venv/bin/pip install --upgrade pip wheel
  venv/bin/pip install -r docker/requirements.txt
  venv/bin/pip install flashinfer-python flashinfer-cubin==0.6.13
fi
ln -sfn /app/qwen-serving/venv /app/venv
ln -sfn /app/qwen-serving/prepare /app/prepare
NVIDIA_NVCC=$(venv/bin/python -c 'import nvidia.cuda_nvcc, os; print(os.path.join(os.path.dirname(nvidia.cuda_nvcc.__file__), "bin"))')
sudo install -d -m 0755 /usr/local/cuda/bin /usr/local/cuda/include
sudo ln -sfn "$NVIDIA_NVCC/nvcc" /usr/local/cuda/bin/nvcc
CURAND_H=$(find venv -name curand.h -print -quit)
[ -n "$CURAND_H" ] || { echo 'curand.h missing from pinned environment' >&2; exit 1; }
sudo ln -sfn "$(realpath "$CURAND_H")" /usr/local/cuda/include/curand.h
rm -rf "$HOME/.cache/flashinfer"
SP=$(venv/bin/python -c 'import vllm, os; print(os.path.dirname(vllm.__file__))')
if ! grep -q dflash2-backport "$SP/vllm/engine/arg_utils.py" 2>/dev/null; then
  for patch_file in patches/*.patch; do patch -p1 -N -d "$SP" < "$patch_file"; done
fi
grep -q dflash2-backport "$SP/vllm/engine/arg_utils.py"
'''
    run(["bash", "-lc", install])
else:
    print("Install skipped in recorded mode.")

## Prepare the pinned model artifacts

The helper uses immutable Hugging Face revisions, performs the same W4A16 preparation, assembles the fast overlay, and fetches the W4A16 DFlash2 draft. Downloads are resumable.

In [ ]:
if RUN_LIVE:
    run([
        str(RUNTIME_ROOT / "venv/bin/python"),
        str(RECIPE_DIR / "prepare_pinned_models.py"),
        "--runtime-root", str(RUNTIME_ROOT),
        "--model-root", str(MODEL_ROOT),
    ])
else:
    print("Model preparation skipped in recorded mode.")

## Start the OpenAI-compatible service

The service uses one card, DFlash2 `k=7`, the fast W4A16 target, BF16 KV, and the measured 65,536-token profile. The API key is generated into an ignored local file and is never printed.

In [ ]:
if RUN_LIVE:
    if not API_SECRET_FILE.exists():
        API_SECRET_FILE.write_text(subprocess.check_output(["openssl", "rand", "-hex", "32"], text=True).strip())
        API_SECRET_FILE.chmod(0o600)
    api_key = API_SECRET_FILE.read_text().strip()
    environment = dict(os.environ)
    environment.update({
        "CUDA_VISIBLE_DEVICES": str(GPU_INDEX),
        "VLLM_API_KEY": api_key,
        "SPEC": "dflash2", "CTX": "fast", "MAX_SEQS": "1",
        "DFLASH_TOKENS": "7", "PORT": str(PORT), "GPU_UTIL": "0.90", "KV_MEM": "",
        "MODEL": str(MODEL_ROOT / "Qwen3.8-27B-W4A16-AutoRound-fast"),
        "DRAFT": str(MODEL_ROOT / "Qwen3.8-27B-DFlash2-W4A16"),
        "VLLM_NO_USAGE_STATS": "1", "DO_NOT_TRACK": "1",
        "FLASHINFER_DISABLE_VERSION_CHECK": "1", "VLLM_V2_CUDAGRAPH_MEM_MIB": "1400",
    })
    log_path = Path("/tmp/pixelml-qwen38-server.log")
    server = subprocess.Popen(["bash", "single-user/start_qwen.sh"], cwd=RUNTIME_ROOT, env=environment, stdout=log_path.open("w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        check = subprocess.run(["curl", "-fsS", "--max-time", "3", f"http://127.0.0.1:{PORT}/health"], capture_output=True)
        if check.returncode == 0:
            print("SERVICE READY")
            break
        if server.poll() is not None:
            raise RuntimeError("service exited before health gate; inspect the local redacted log")
        time.sleep(10)
    else:
        raise TimeoutError("service did not become ready within 30 minutes")
else:
    print("Service start skipped in recorded mode.")

## Benchmark and recorded output

The live path runs the pinned usage-token-counted harness. Publication is blocked if the server omits `usage.completion_tokens`; SSE events are never counted as tokens.

In [3]:
if RUN_LIVE:
    evidence_root = Path("/tmp/pixelml-qwen38-evidence")
    if not (evidence_root / ".git").exists():
        run(["git", "clone", "https://github.com/PixelML/Qwen3.8-27B-CMP-170HX", str(evidence_root)])
    run(["git", "-C", str(evidence_root), "checkout", "--detach", "41d2c414fe0f293d77087ef18cda5896664754d6"])
    harness = runpy.run_path(str(evidence_root / "scripts/bench-usage.py"), run_name="pixelml_bench")
    benchmark = harness["Benchmark"](f"http://127.0.0.1:{PORT}", API_SECRET_FILE.read_text().strip())
    prompt_tokens = {"story": benchmark.token_count(harness["P256"]), "long": benchmark.token_count(harness["LONG"])}
    sampler = harness["TelemetrySampler"](GPU_INDEX, Path("/tmp/pixelml-qwen38-telemetry.jsonl"))
    live_results = Path("/tmp/pixelml-qwen38-live.jsonl")
    sampler.start()
    try:
        with live_results.open("w") as handle, contextlib.redirect_stdout(handle):
            benchmark.run_case("decode256", harness["P256"], 256, prompt_tokens["story"])
            sampler.assert_safe()
            benchmark.run_case("decode900", harness["P256"], 900, prompt_tokens["story"])
            sampler.assert_safe()
            benchmark.run_case("prefill_long", harness["LONG"], 8, prompt_tokens["long"])
            sampler.assert_safe()
    finally:
        sampler.stop()
    print(f"Live result written to {live_results}")

rows = list(csv.DictReader((RECIPE_DIR / "results/summary.csv").open()))
columns = ["card", "decode_256_tok_s", "decode_900_tok_s", "prefill_6603_tok_s", "ttft_ms"]
print(" | ".join(columns))
print(" | ".join(["---"] + ["---:"] * (len(columns) - 1)))
for row in rows:
    print(" | ".join(row[column] for column in columns))

card | decode_256_tok_s | decode_900_tok_s | prefill_6603_tok_s | ttft_ms
--- | ---: | ---: | ---: | ---:
card-0 | 135.31 | 121.28 | 1957.3 | 201.2
card-1 | 140.27 | 124.78 | 1954.7 | 189.7
card-2 | 133.57 | 119.94 | 1926.0 | 181.4
mean | 136.38 | 122.00 | 1946.0 | 190.8

In [4]:
if RUN_LIVE:
    try:
        import PIL  # noqa: F401
    except ImportError:
        run([sys.executable, "-m", "pip", "install", "Pillow"])
run([
    sys.executable, str(REPO_ROOT / "scripts/render_recipe_chart.py"),
    "--spec", str(RECIPE_DIR / "chart-spec.json"),
    "--output", str(RECIPE_DIR / "assets/performance.png"),
])
print("Chart regenerated from committed results: assets/performance.png")

Chart regenerated from committed results: assets/performance.png

## Try your own prompt

Edit `PROMPT` in the configuration cell or below. With the service ready, this final `curl` prints the model response and the authoritative final usage object.

In [ ]:
PROMPT = "Write a compact Python function that validates a topological ordering. Return code only."
os.environ["PIXELML_PROMPT"] = PROMPT
os.environ["PIXELML_PORT"] = str(PORT)
os.environ["PIXELML_API_SECRET_FILE"] = str(API_SECRET_FILE)
print(PROMPT)

In [ ]:
%%bash
set -euo pipefail
if [ "${PIXELML_RUN_LIVE:-0}" != "1" ]; then
  echo "Set PIXELML_RUN_LIVE=1, rerun from Configure, then run this cell."
  exit 0
fi
BODY=$(jq -n --arg prompt "$PIXELML_PROMPT" '{model:"qwen3.8-27b",prompt:$prompt,max_tokens:256,temperature:0.0,stream:false}')
RESPONSE=$(curl -sS "http://127.0.0.1:$PIXELML_PORT/v1/completions" \
  -H "Authorization: Bearer $(<"$PIXELML_API_SECRET_FILE")" \
  -H 'Content-Type: application/json' \
  -d "$BODY")
printf '%s\n' "$RESPONSE" | jq -r '.choices[0].text'
printf '\nusage:\n'
printf '%s\n' "$RESPONSE" | jq '.usage | {prompt_tokens, completion_tokens, total_tokens}'